# NCM GPT-1 在 Kaggle 訓練（T4 × 2，約 30 GPU 小時）

設定（右上角）：
- Accelerator: **GPU T4 x2**
- Internet: **On**
- 用 **Save & Run All**（不要用互動模式長跑，閒置約 20 分鐘會被中斷）

每一場最多 12 小時。不要重裝 PyTorch。
跑完後把 Output 加成下一場的 Input dataset，同一個 notebook 再跑一次即可續訓。

T4（sm_75）上 generator 在每卡 batch 8 與 batch 4、且未開梯度檢查點時會 OOM。此 notebook 預設每卡 batch 2，並開啟梯度檢查點與 `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`。Ampere 以前的 GPU 會自動改用 fp16（不要用模擬的 bf16）。


In [ ]:
# 只補訓練依賴，不要重裝 torch
import os, subprocess, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "omegaconf", "hydra-core", "datasets", "regex", "tqdm"])
print("deps ok")

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO = "https://github.com/nicholashui/ncm-gpt1.git"
ROOT = Path("/kaggle/working/ncm-gpt1")
if not (ROOT / "train_tokenizer.py").exists():
    # 若已把程式當成 Input dataset 掛進來，直接複製；否則 clone
    local = None
    for cand in Path("/kaggle/input").glob("*/train_tokenizer.py") if Path("/kaggle/input").exists() else []:
        local = cand.parent
        break
    if local:
        import shutil
        shutil.copytree(local, ROOT, dirs_exist_ok=True)
        print("copied repo from", local)
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(ROOT)])
        print("cloned", REPO)
else:
    print("repo already present")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("cwd", Path.cwd())

In [ ]:
import torch
print("cuda:", torch.cuda.is_available(), "gpus:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        cap = torch.cuda.get_device_capability(i)
        print(i, torch.cuda.get_device_name(i), "mem", round(torch.cuda.get_device_properties(i).total_memory/1024**3, 1), "GiB", "cap", cap)
assert torch.cuda.is_available(), "請把 Accelerator 設成 GPU T4 x2"
assert torch.cuda.device_count() >= 1

訓練語料預設是開源 BookCorpus 重建版（與 2018 GPT-1 的書籍語料同一類）。若下載失敗，腳本會改用 MiniPile，再不行才用 TinyStories。檢查點寫到 `/kaggle/working/artifacts`。

若上一場 Output 已掛成 Input，且 tokenizer 已完成，就會跳過 tokenizer、直接續訓 generator。


In [ ]:
import os, subprocess, sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

out = Path("/kaggle/working/artifacts")
out.mkdir(parents=True, exist_ok=True)

# 若上一場的 Output 已掛成 Input，自動接續
cmd = [
    sys.executable, "scripts/train_kaggle.py",
    "--hours", "11.5",
    "--reserve_minutes", "15",
    "--output_dir", str(out),
    "--config", "config/kaggle.yaml",
    "--batch_size", "2",
    "--gradient_checkpointing",
]
print("running:", " ".join(cmd), flush=True)
rc = subprocess.call(cmd)
print("exit", rc)
if rc != 0:
    raise SystemExit(rc)

In [ ]:
from pathlib import Path

def show(stage):
    p = Path("/kaggle/working/artifacts") / stage / "status.json"
    print("=" , stage, "=")
    if p.exists():
        print(p.read_text())
    else:
        print("no status.json yet")
    log = Path("/kaggle/working/artifacts") / stage / f"{stage}_log.jsonl"
    if log.exists():
        lines = log.read_text(encoding="utf-8").strip().splitlines()
        print("log lines", len(lines))
        if lines:
            print("last:", lines[-1])

show("tokenizer")
show("generator")